# 🎙️ TontumaBot V3 — Oolel-Voices TTS (Google Colab)

Ce notebook fait tourner **Oolel-Voices** (soynade-research) sur Colab,
qui dispose de PyTorch >= 2.1.1 nécessaire pour SDPA.

**Architecture retenue :**
- Local (V3) → SpeechT5 wolof (`bilalfaye/speecht5_tts-wolof-v0.2`)
- Colab → Oolel-Voices (`soynade-research/Oolel-Voices`) — meilleure qualité vocale

---

## 1. Vérification de l'environnement

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

# Vérifier que torch >= 2.1.1 (requis par Oolel-Voices)
from packaging import version
assert version.parse(torch.__version__) >= version.parse('2.1.1'), \
    f'PyTorch >= 2.1.1 requis, version actuelle : {torch.__version__}'
print('✅ Version PyTorch compatible avec Oolel-Voices')

## 2. Installation des dépendances

In [ ]:
!pip install -q huggingface_hub soundfile einops conformer==0.3.2 omegaconf diffusers==0.29.0 scipy "numpy<2.0"

## 3. Téléchargement du modèle Oolel-Voices

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import sys

REPO_ID  = 'soynade-research/Oolel-Voices'
ckpt_dir = Path(snapshot_download(repo_id=REPO_ID))
print('✅ Oolel-Voices téléchargé :', ckpt_dir)

# Ajouter au path pour importer modeling_oolel_voices
if str(ckpt_dir) not in sys.path:
    sys.path.insert(0, str(ckpt_dir))

## 4. Chargement du modèle

In [ ]:
from modeling_oolel_voices import OolelVoicesForInference

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Chargement sur :', device)

model = OolelVoicesForInference.from_pretrained(str(ckpt_dir), device_map=device)
model.eval()
print('✅ Oolel-Voices prêt (sample rate :', model.sr, 'Hz)')

# Chercher le voice prompt de référence
voice_prompt = None
for candidate in [ckpt_dir / '8_1_c.wav']:
    if candidate.exists():
        voice_prompt = str(candidate)
        print('Voice prompt :', voice_prompt)
        break
if not voice_prompt:
    print('⚠️  Pas de voice prompt — synthèse sans clonage de voix')

## 5. Fonction de synthèse

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

def synthesize(text: str, out_path: str = 'output.wav',
               exaggeration: float = 0.5,
               cfg_weight: float   = 0.5,
               temperature: float  = 0.8) -> str:
    """Synthétise le texte avec Oolel-Voices et sauvegarde en WAV."""
    kwargs = {}
    if voice_prompt:
        kwargs['audio_prompt_path'] = voice_prompt

    print(f'Synthèse : "{text[:80]}"...')
    wav = model.generate(
        text,
        exaggeration = exaggeration,
        cfg_weight   = cfg_weight,
        temperature  = temperature,
        **kwargs,
    )
    audio_np = wav.squeeze(0).detach().cpu().numpy()
    sf.write(out_path, audio_np, model.sr, format='WAV')
    print(f'✅ Audio sauvegardé : {out_path} ({len(audio_np)/model.sr:.2f}s)')
    display(Audio(out_path))
    return out_path

## 6. Tests — Français et Wolof

In [ ]:
# Test français
synthesize(
    'Bonjour, je suis TontumaBot, votre assistant administratif au Sénégal.',
    out_path='test_fr.wav'
)

In [ ]:
# Test wolof (réponse traduite par le pipeline V3)
synthesize(
    'Salam aléikum. Dama dem ci maternité bi ngir jëfandikoo këyit juddu gi.',
    out_path='test_wo.wav'
)

In [ ]:
# Test réponse administrative complète
synthesize(
    'Pour obtenir un extrait de naissance, rendez-vous à la mairie de votre lieu de naissance '
    'avec le carnet de famille et une pièce d identité.',
    out_path='test_admin.wav'
)

## 7. Intégration avec le pipeline TontumaBot V3

Pour utiliser Oolel-Voices comme TTS dans le pipeline complet :

In [ ]:
# Optionnel : monter Google Drive pour accéder aux fichiers V3
# from google.colab import drive
# drive.mount('/content/drive')
# V3_PATH = '/content/drive/MyDrive/TontumaBot_RAG1/V3'

# Pipeline complet : question → RAG → LLM → TTS Oolel
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', f'{V3_PATH}/requirements.txt', '-q'])
# sys.path.insert(0, f'{V3_PATH}/src')
# sys.path.insert(0, f'{V3_PATH}/data')
# from pipeline import answer
#
# result = answer(
#     'dama beug wout kayitu juddu ?',
#     provider='groq',
#     tts=True,
#     tts_engine='oolel',
#     tts_out='/content/response.wav'
# )
# display(Audio('/content/response.wav'))
# print('Réponse WO:', result['response_wo'])

print('Pipeline complet : décommenter les lignes ci-dessus après avoir monté Drive')

---
## Notes

| Paramètre | Valeur recommandée | Effet |
|-----------|-------------------|-------|
| `exaggeration` | 0.3 – 0.6 | Expressivité de la voix |
| `cfg_weight` | 0.3 – 0.7 | Adhérence au voice prompt |
| `temperature` | 0.6 – 0.9 | Variabilité (0.8 = équilibre) |

**Oolel-Voices** utilise un mécanisme de **voice cloning** à partir du fichier audio de référence `8_1_c.wav`.
Pour utiliser une autre voix de référence, fournir un fichier WAV de 3 à 10 secondes à `audio_prompt_path`.